# This script contains reusable code for both Batch and Streaming processing of JSON Invoices

In [0]:
class invoiceStreamBatch:
    
    def __init__(self):
        self.data_path='/Volumes/cat_spark_streaming/jsonstream/dataset/json-datasets/'
        self.checkpoint_path='/Volumes/cat_spark_streaming/jsonstream/checkpoint_dir'
        self.catalog_name='cat_spark_streaming'
        self.schema_name='jsonstream'
        self.table_name='invoice'


    def get_schema(self):
        from pyspark.sql.types import StringType,FloatType,ArrayType,StructType,IntegerType,StructField
        json_schema= StructType([
                    StructField("InvoiceNumber", StringType()),
                    StructField('CreatedTime',StringType()),
                    StructField('StoreID', StringType()),
                    StructField('PosID',StringType()),
                    StructField('CashierID',StringType()),
                    StructField('CustomerType', StringType()),
                    StructField('CustomerCardNo', StringType()),
                    StructField('TotalAmount', FloatType()),
                    StructField('NumberOfItems', IntegerType()),
                    StructField('PaymentMethod', StringType()),
                    StructField('TaxableAmount', FloatType()),
                    StructField('CGST', FloatType()),
                    StructField('SGST', FloatType()),
                    StructField('CESS',  FloatType()),
                    StructField('DeliveryType', StringType()),
                    StructField('DeliveryAddress', StructType([
                        StructField('AddressLine', StringType()),
                        StructField('City', StringType()),
                        StructField('State', StringType()),
                        StructField('PinCode', StringType()),
                        StructField('ContactNumber', StringType())
                    ])),
                    StructField('InvoiceLineItems', ArrayType(
                        StructType([
                            StructField('ItemCode', StringType()),
                            StructField('ItemDescription', StringType()),
                            StructField('ItemPrice', FloatType()),
                            StructField('ItemQty',IntegerType()),
                            StructField('TotalValue',FloatType())
                        ])
                    ))
                ])
        
        return json_schema
    

    def read_raw_data(self):
        return(
                spark.readStream
                    .format('json')
                    .schema(self.get_schema())
                    .load(self.data_path) 
        )

    
    def json_transformation(self,df):
        from pyspark.sql.functions import explode_outer,col
        return (
                df.select(
                    'InvoiceNumber',
                    'CreatedTime',
                    'StoreID',
                    'PosID',
                    'CashierID',
                    'CustomerType',
                    'CustomerCardNo',
                    'TotalAmount',
                    'NumberOfItems',
                    'PaymentMethod',
                    'TaxableAmount',
                    'CGST',
                    'SGST',
                    'CESS',
                    'DeliveryType',
                    'DeliveryAddress.AddressLine',
                    'DeliveryAddress.City',
                    'DeliveryAddress.State',
                    'DeliveryAddress.PinCode',
                    'DeliveryAddress.ContactNumber',
                    explode_outer(col('InvoiceLineItems')).alias('InvoiceLineItems'))
                        .select( '*',
                                'InvoiceLineItems.ItemCode',
                                'InvoiceLineItems.ItemDescription',
                                'InvoiceLineItems.ItemPrice',
                                'InvoiceLineItems.ItemQty',
                                'InvoiceLineItems.TotalValue'
            ).drop('InvoiceLineItems')
        )


    def write_to_table(self,df,trigger="batch"):
        query=(
                df.writeStream
                    .format('delta')
                    .outputMode('append')
                    .option('checkpointLocation',self.checkpoint_path)
                    .option('maxFilesPerTrigger',2)
        )
        
        if trigger=="batch":
            return (query.trigger(availableNow=True)
                        .toTable(f'{self.catalog_name}.{self.schema_name}.{self.table_name}') 
            )
        else:
            return (query.trigger(processingTime=trigger)
                        .toTable(f'{self.catalog_name}.{self.schema_name}.{self.table_name}') 
                    )

        query.awaitTermination()
        

    def processing(self,trigger="batch"):
        print(f'\tStarting Invoice Processing...', end='')
        raw_df=self.read_raw_data()
        transformed_df=self.json_transformation(raw_df)
        self.write_to_table(transformed_df,trigger)     